# recursive-flow coding-agent walkthrough

Build a recursive coding agent, run a task, save the run, and embed the generated artifact inline. This mirrors `examples/coding/agent.py` but as a notebook so you can poke at every step.

Sibling notebooks read offline against a saved run directory under `examples/_runs/word-search/baseline/`:

- [`node_basics.ipynb`](./node_basics.ipynb) — querying a run: `graph.walk`, `graph.find`, `graph.where`, `Graph.load`, …
- [`viz_walkthrough.ipynb`](./viz_walkthrough.ipynb) — visualizing a run: `tree`, `gantt`, `code_log`, mermaid / dot / d2, `report_md`, the inline Gradio viewer, etc.

Running this notebook end-to-end requires `OPENAI_API_KEY` and live LLM calls. Skip ahead to the other notebooks if you just want to consume a saved run.

## 1. Build the agent

The whole run is one `Flow` driving one `Graph`:

- `Flow` — wires an LLM client (plus optional cheaper alternates registered as `fast`) to a stateful REPL and exposes `start` / `step`. The whole tree advances by synchronized ticks; `flow.start(query)` seeds the run and returns the `Graph`, then `graph = flow.step(graph)` advances one tick and returns a fresh snapshot.
- `Runtime` — *where* code runs and *which* tools it has. A `LocalRuntime` runs in-process; `runtime.register_tools(FILE_TOOLS)` exposes the filesystem tools (`read_file`, `write_file`, `edit_file`, `ls`, `grep`, …) to every agent and child — no `Flow` subclass.
- `working_directory` — set it on the runtime and agent code (and the file tools) run inside it; no manual `chdir`. Swap `LocalRuntime` for `DockerRuntime(image, working_directory=...)` to sandbox each step with the same interface.

In [2]:
from pathlib import Path

import rflow
from rflow.tools import FILE_TOOLS


def build_agent(
    workdir: Path | str,
    max_depth: int = 3,
    max_iters: int = 30,
    max_concurrency: int = 8,
) -> rflow.Flow:
    """Construct a coding agent identical to examples/coding/agent.py."""
    # The runtime owns where code runs and which tools it has. LocalRuntime runs
    # in-process with the cwd switched into workdir; register_tools exposes the
    # file tools to every agent and child. Swap in DockerRuntime to sandbox.
    runtime = rflow.LocalRuntime(working_directory=workdir)
    runtime.register_tools(FILE_TOOLS)
    return rflow.Flow(
        rflow.OpenAIClient("gpt-5"),
        llm_clients={"fast": rflow.OpenAIClient("gpt-5-mini")},
        runtime=runtime,
        max_depth=max_depth,
        max_iters=max_iters,
        max_concurrency=max_concurrency,
    )

## 2. Run a task

`agent.start(query)` seeds and returns the live `Graph` (just the root agent with its `UserQuery` node). Each `graph = agent.step(graph)` advances exactly one tick — one LLM call, one REPL execution, or a wait-resolution — and returns a fresh, frozen `Graph` snapshot (the graph you pass stays untouched, so each returned value is a checkpoint you can keep). Use `graph.current()` to inspect the latest node, `graph.finished` to stop, `graph.result()` for the terminal answer; append each returned `graph` to a list for a timeline.

`live_view()` opens a self-updating Rich tree; you own the loop and just hand it the latest `Graph` on each tick. The agent loop and the visualization are decoupled.

In [3]:
TASK = """Create a runnable browser-based boids simulation in plain HTML, CSS, and JavaScript.

Requirements:
- The main runnable interface is `index.html`.
- Write separate files for:
    - `index.html`
    - `style.css`
    - `boids.js`
- Do not use build tools or external libraries.
- Use a dark color background.
- Do not use ES modules; wire scripts with `<script src="..."></script>` tags.
- Render 100s of colorful boids on a 2D canvas. Do not add configurations, just the canvas.
- Verify that all files exist, script tags are ordered correctly, and the JavaScript has no obvious syntax/runtime wiring errors before returning.
"""

WORKDIR = Path("../_runs/notebooks/boids-sim").resolve()
WORKDIR.mkdir(parents=True, exist_ok=True)

agent = build_agent(WORKDIR, max_depth=2)

In [ ]:
graph = agent.start(TASK)
graphs = [graph.copy(deep=True)]
# Built from the live prompt builder so the preview tracks prompt changes.
prompt = agent.build_system_prompt(graph)
print(prompt)

In [ ]:
from rflow.utils.viz import live_view

# The runtime already runs agent code inside WORKDIR — no chdir needed here.
with live_view() as view:
    view(graph)
    while not graph.finished:
        graph = agent.step(graph)  # returns a fresh, frozen snapshot each tick
        graph.save(WORKDIR / "graph")
        view(graph)

In [ ]:
current = graph.current()
print(f"{len(graphs)} snapshots  \u00b7  final: {graph.agent_id} [{current.type}]")
print(f"query : {graph.query[:120]!r}...")
print(f"result: {graph.result()[:200]}")

In [ ]:
print(graph.tree())

In [ ]:
# Persist the run as a run directory (manifest + nested per-agent logs),
# right alongside the files the agent wrote into WORKDIR.
run_dir = graph.save(WORKDIR / "graph")
print(f"run -> {run_dir}  ({len(graph.agents)} agents)")

# Or keep the full stepped timeline as a trace.json:
# trace_path = rflow.save_trace(graphs, WORKDIR / "trace.json", metadata={"task": TASK})
# print(f"trace -> {trace_path}  ({len(graphs)} snapshots)")

## 3. Preview the generated artifact

Serve the generated `index.html` over local HTTP and embed that URL in the notebook. This exercises the same browser module-loading rules as opening the artifact normally, so import/export mistakes are not hidden by `srcdoc` inlining.

In [ ]:
from functools import partial
from http.server import SimpleHTTPRequestHandler, ThreadingHTTPServer
from IPython.display import IFrame
import socket
import threading

candidates = sorted(WORKDIR.glob("**/index.html"))
if not candidates:
    raise FileNotFoundError(f"no index.html under {WORKDIR}")

html_path = candidates[-1].resolve()
site_root = html_path.parent

if "preview_server" in globals():
    preview_server.shutdown()

with socket.socket() as s:
    s.bind(("127.0.0.1", 0))
    port = s.getsockname()[1]

class QuietHandler(SimpleHTTPRequestHandler):
    def log_message(self, *args):
        pass

handler = partial(QuietHandler, directory=str(site_root))
preview_server = ThreadingHTTPServer(("127.0.0.1", port), handler)
threading.Thread(target=preview_server.serve_forever, daemon=True).start()

url = f"http://127.0.0.1:{port}/{html_path.name}"
print(f"serving {site_root} -> {url}")
IFrame(url, width="100%", height=600)

## 4. Open the interactive viewer

`open_viewer(source)` boots a Gradio stepper (step slider + clickable graph + per-agent transcript) inline.

In [4]:
from rflow.utils.viewer import open_viewer

open_viewer(WORKDIR / "graph", inline=True, height=720, quiet=True)

## 4. Render frames of each step

In [ ]:
# recursive-flow render ../_runs/notebooks/boids-sim \
#   --format steps \
#   --out ../_runs/notebooks/boids-sim/frames \
#   --width 1600 \
#   --height 1200 \
#   --scale 2

In [ ]:
from rflow.utils.viewer import save_steps

save_steps(
    WORKDIR,
    WORKDIR / "frames",
    width=1600,
    height=1200,
    scale=2,
)

## Next

- [`node_basics.ipynb`](./node_basics.ipynb) — walk the saved trace with the `Graph` query API.
- [`viz_walkthrough.ipynb`](./viz_walkthrough.ipynb) — inline Plotly, Mermaid, DOT, Gantt, report exports.